In [1]:
# Extract origin AS from AS Path that could be of different formats: AS set, confederation set/sequence
import re

def get_origin(as_path):
    global elements
    # Match different possible formats using regular expressions
    # Simple format
    if re.match(r"^\d+(\s\d+)*$", as_path):
        # Format: "100 200 300"
        elements = as_path.split()
        
    # AS set
    elif re.match(r"^\{\d+(,\d+)*\}$", as_path):
        # Format: "{36040,38266,45271,55410}"
        elements = as_path.strip("{}").split(',')
    
    # AS Confederation Set
    elif re.match(r"^\[\d+(,\d+)*\]$", as_path):
        # Format: "[12345,6789]"
        elements = as_path.strip("[]").split(',')
    
    # AS Confederation Sequence 
    elif re.match(r"^\(\d+(\s\d+)*\)$", as_path):
        # Format: "(12345 6789)"
        elements = as_path.strip("()").split()

    # Return the last element in the parsed list
    return elements[-1]

In [3]:
# Detects if a given string is an IPv4 or IPv6
import ipaddress

def detect_ip_version(ip_network_str):
    try:
        ip_network = ipaddress.ip_network(ip_network_str, strict=False)
        if isinstance(ip_network, ipaddress.IPv4Network):
            return "IPv4"
        elif isinstance(ip_network, ipaddress.IPv6Network):
            return "IPv6"
    except ValueError:
        return "Invalid IP address or network"


In [ ]:
#!/usr/bin/env python
# Get live data from routeviews and find submoas
import pybgpstream
from collections import defaultdict
import pytricia
from ipaddress import IPv6Address, IPv6Network, IPv4Network

stream = pybgpstream.BGPStream(
    # accessing routeview-stream
    project="routeviews-stream",
    # filter to show only stream from amsix bmp stream
    filter="router amsix"
)

# Build a Patricia tree
pyt_v6 = pytricia.PyTricia(128) # 128 is used for storing IPv6
pyt_v4 = pytricia.PyTricia() # For storing IPv4

submoas_v4 = defaultdict(set) # Storing subMOAS for IPv4
submoas_v6 = defaultdict(set) # Storing subMOAS for IPv6


for elem in stream:
    #print("Elem is %s" %(elem.fields))
    # Sample stream data
    # stream = [{}]
    # <prefix, origin-ASns-set > dictionary

#     # Get prefix and origin
    for elem in stream:
        pfx = elem.fields["prefix"]
        
#         print("Prefix is ", pfx)
        
        # Ignore default routes and withdrawal messages 
        if pfx != '0.0.0.0/0' and "as-path" in elem.fields:
            as_path = elem.fields["as-path"]
            # Get the list of ASes in the AS path
            orig = get_origin(as_path)            orig = get_origin(as_path)

#             print("Prefix is %s, origin is %s" %(pfx, orig))
            
            # Check if the prefix is IPv4 or IPv6 as object has to be stored accordingly
            if detect_ip_version(pfx) == 'IPv4':
                pyt_v4.insert(pfx, orig)
                # Check if parent exists for this prefix and the origins are different
                if pyt_v4.parent(pfx):
                    old_orig = pyt_v4[pyt_v4.parent(pfx)]

                    # Check if two origins are different TODO: and they are from different organization
                    if old_orig != orig:
                        print("#####New sub prefix is %s with origin %s and parent prefix is %s and origin %s, " %(pfx, orig, pyt_v4.parent(pfx),old_orig ))
                        submoas_v4[pfx].add(orig)
                        print("Count submoas %s" %(len(submoas_v4)))

                        #TODO: Check legitimate owner of the prefix from IRR and RPKI dataset
                    del pyt_v4[pfx]

                # Check if spefic prefix(child) already exists    
                elif pyt_v4.children(pfx):
                    # print("Child exists")
                    for child in pyt_v4.children(pfx):

                        old_orig = pyt_v4[child]

                        # Check if two origins are different TODO: and they are from different organization
                        if old_orig != orig:
                            print("@@@@New prefix %s with origin %s has a child prefix  %s with origin %s, " %(pfx, orig, child, old_orig ))
                            submoas_v4[pfx].add(orig)
                            print("Count submoas %s" %(len(submoas_v4)))
                        del pyt_v4[child]
                        
                        
                        
            else:
                pyt_v6.insert(pfx, orig)

                # Check if parent exists for this prefix and the origins are different
                if pyt_v6.parent(pfx):
                    old_orig = pyt_v6[pyt_v6.parent(pfx)]

                    # Check if two origins are different TODO: and they are from different organization
                    if old_orig != orig:
                        print("#####New sub prefix is %s with origin %s and parent prefix is %s and origin %s, " %(pfx, orig, pyt_v6.parent(pfx),old_orig ))
                        submoas_v6[pfx].add(orig)
                        print("Count submoas %s" %(len(submoas_v6)))

                        #TODO: Check legitimate owner of the prefix from IRR and RPKI dataset
                    del pyt_v6[pfx]

                # Check if spefic prefix(child) already exists    
                elif pyt_v6.children(pfx):
                    # print("Child exists")
                    for child in pyt_v6.children(pfx):

                        old_orig = pyt_v6[child]

                        # Check if two origins are different TODO: and they are from different organization
                        if old_orig != orig:
                            print("@@@@New prefix %s with origin %s has a child prefix  %s with origin %s, " %(pfx, orig, child, old_orig ))
                            submoas_v6[pfx].add(orig)
                            print("Count submoas %s" %(len(submoas_v6)))
                        del pyt_v6[child]
#             print("Count submoas %s" %(len(submoas)))      

In [ ]:
submoas_v4

In [7]:
#!/usr/bin/env python
# Get one second of historical data from routeviews and find submoas
import pybgpstream
from collections import defaultdict
import pytricia
# from ipaddress import IPv6Address, IPv6Network, IPv4Network

stream = pybgpstream.BGPStream(
    # Consider this time interval:
    # Sat, 01 Aug 2015 7:50:00 GMT -  08:10:00 GMT
    from_time="2024-06-12 00:00:00", until_time="2024-06-12 00:00:01",
    collectors=["rrc00"],
    record_type="ribs",
    #filter = "origin 1140"
)

# Build a Patricia tree
pyt_v6 = pytricia.PyTricia(128) # 128 is used for storing IPv6
pyt_v4 = pytricia.PyTricia() # For storing IPv4

submoas_v4 = defaultdict(set) # Storing subMOAS for IPv4
submoas_v6 = defaultdict(set) # Storing subMOAS for IPv6


for elem in stream:
    #print("Elem is %s" %(elem.fields))
    # Sample stream data
    # stream = [{}]
    # <prefix, origin-ASns-set > dictionary

#     # Get prefix and origin
    for elem in stream:
        pfx = elem.fields["prefix"]
        
#         print("Prefix is ", pfx)
        
        # Ignore default routes and withdrawal messages 
        if pfx != '0.0.0.0/0' and "as-path" in elem.fields:
            as_path = elem.fields["as-path"]
            # Get the list of ASes in the AS path
            orig = get_origin(as_path)
#             print("Prefix is %s, origin is %s" %(pfx, orig))
            
            # Check if the prefix is IPv4 or IPv6 as object has to be stored accordingly
            if detect_ip_version(pfx) == 'IPv4':
                pyt_v4.insert(pfx, orig)
                # Check if parent exists for this prefix and the origins are different
                if pyt_v4.parent(pfx):
                    old_orig = pyt_v4[pyt_v4.parent(pfx)]

                    # Check if two origins are different TODO: and they are from different organization
                    if old_orig != orig:
#                         print("#####New sub prefix is %s with origin %s and parent prefix is %s and origin %s, " %(pfx, orig, pyt_v4.parent(pfx),old_orig ))
                        submoas_v4[pfx].add(orig)
#                         print("Count submoas %s" %(len(submoas_v4)))

                        #TODO: Check legitimate owner of the prefix from IRR and RPKI dataset
                    del pyt_v4[pfx]

                # Check if spefic prefix(child) already exists    
                elif pyt_v4.children(pfx):
                    # print("Child exists")
                    for child in pyt_v4.children(pfx):

                        old_orig = pyt_v4[child]

                        # Check if two origins are different TODO: and they are from different organization
                        if old_orig != orig:
                            print("@@@@New prefix %s with origin %s has a child prefix  %s with origin %s, " %(pfx, orig, child, old_orig ))
                            submoas_v4[pfx].add(orig)
                            print("Count submoas %s" %(len(submoas_v4)))
                        del pyt_v4[child]
                        
                        
                        
            else:
                pyt_v6.insert(pfx, orig)

                # Check if parent exists for this prefix and the origins are different
                if pyt_v6.parent(pfx):
                    old_orig = pyt_v6[pyt_v6.parent(pfx)]

                    # Check if two origins are different TODO: and they are from different organization
                    if old_orig != orig:
#                         print("#####New sub prefix is %s with origin %s and parent prefix is %s and origin %s, " %(pfx, orig, pyt_v6.parent(pfx),old_orig ))
                        submoas_v6[pfx].add(orig)
#                         print("Count submoas %s" %(len(submoas_v6)))

                        #TODO: Check legitimate owner of the prefix from IRR and RPKI dataset
                    del pyt_v6[pfx]

                # Check if spefic prefix(child) already exists    
                elif pyt_v6.children(pfx):
                    # print("Child exists")
                    for child in pyt_v6.children(pfx):

                        old_orig = pyt_v6[child]

                        # Check if two origins are different TODO: and they are from different organization
                        if old_orig != orig:
#                             print("@@@@New prefix %s with origin %s has a child prefix  %s with origin %s, " %(pfx, orig, child, old_orig ))
                            submoas_v6[pfx].add(orig)
#                             print("Count submoas %s" %(len(submoas_v6)))
                        del pyt_v6[child]
#             print("Count submoas %s" %(len(submoas)))   

        

In [11]:
len(submoas_v6)

220592

In [ ]:
# Dump MS annoucement into a graph object
import pickle

# Dump submoas object into pickle form
with open('submoas1.p', 'wb') as pickleFile:
    pickle.dump(submoas, pickleFile)


In [ ]:
import pickle
# Load pickle file
file = open("submoas1.p",'rb')
submoas = pickle.load(file)
file.close()

In [ ]:
submoas

In [ ]:
# A sample program to find subMOAS from a given list of prefixes
import pytricia
from collections import defaultdict

pyt = pytricia.PyTricia()

# Insert prefix and ASN as pytricia object in a Patricia tree
pfx_origin_list = [{"prefix": "172.32.10.0/24", "origin": "100"}, 
                   {"prefix": "172.32.10.0/25", "origin": "200"}, 
                   {"prefix": "172.32.10.0/26", "origin": "200"}, 
                   {"prefix": "172.33.10.0/24", "origin": "300"},
                   {"prefix": "172.16.0.0/8", "origin": "100"},
                   {"prefix": "0.0.0.0/8", "origin": "1000"}

                  ]
for po in pfx_origin_list:
    pyt.insert(po["prefix"], po["origin"])

print("INPUT")
for p in pyt:
    print("Prefix is %s and origin is %s"  % (p, pyt[p]))

    
# Traverse the tree to find out any prefix that is a subnet of any other
submoas=[]
# <prefix, origin-ASns-set > dictionary
submoas = defaultdict(set)

for p in pyt:
  
    if pyt.parent(p):
#         print("Parent exists for prefix %s with origin %s and parent is %s with origin %s"
#               %(p, pyt[p], pyt.parent(p), pyt[pyt.parent(p)]))
        origin = pyt[p]
        submoas[p].add(origin)  
        
# Print the list of subMOAS prefix and their origin ASns
print("\nOUTPUT SubMOAS are")
for pfx in submoas:
    print("Prefix is %s and origin %s"% (pfx, submoas[pfx]))
    

In [ ]:
#!/usr/bin/env python
# Get live data from RIS Live

import pybgpstream
stream = pybgpstream.BGPStream(
    # accessing ris-live
    project="ris-live",
    # filter to show only stream from rrc00
    filter="collector rrc00",
)

for elem in stream:
    print(elem)

In [ ]:
# This code is from CAIDA BGPStream example which generates MOAS
#!/usr/bin/env python

from collections import defaultdict
import pybgpstream

stream = pybgpstream.BGPStream(
    # Consider this time interval:
    # Sat, 01 Aug 2015 7:50:00 GMT -  08:10:00 GMT
    from_time="2024-06-12 00:00:00", until_time="2024-06-12 23:59:00",
    collectors=["rrc00"],
    record_type="ribs",
    #filter = 'path _13335' # AS13335 is Cloudflare
)

# <prefix, origin-ASns-set > dictionary
prefix_origin = defaultdict(set)

for rec in stream.records():
    for elem in rec:
        # Get the prefix
        pfx = elem.fields["prefix"]
        #print("Elem is ", elem)
        # Get the list of ASes in the AS path
        ases = elem.fields["as-path"].split(" ")
        if len(ases) > 0:
            # Get the origin ASn (rightmost)
            origin = ases[-1]
            # Insert the origin ASn in the set of
            # origins for the prefix
            prefix_origin[pfx].add(origin)
            
            # TODO: Get element time as well 
            # elem.time    

# Print the list of MOAS prefix and their origin ASns
for pfx in prefix_origin:
    if len(prefix_origin[pfx]) > 1:
#         print("list is ", prefix_origin[pfx])
        # Check if one of the origins of MOAS is AS13335(Cloudflare)
        if '113355' in prefix_origin[pfx] or '395747' in prefix_origin[pfx] or '202623' in prefix_origin[pfx] or '202623' in prefix_origin[pfx] or '209242' in prefix_origin[pfx] or '203898' in prefix_origin[pfx] or '139242' in prefix_origin[pfx] or '132892' in prefix_origin[pfx]:
            print((pfx, ",".join(prefix_origin[pfx])))



In [ ]:
# Playing with pytricia: a python librabry to implement prefix radix tree
# https://ttl255.com/subnet-filtering-with-python/
from typing import Iterable, List

import pytricia


def filter_out_subnets(prefixes: Iterable[str], ipv6=False) -> List[str]:
    """
    Goes through prefixes and filters out any prefix that is a subnet of any other
    Uses Patricia Tree for efficient prefix lookups
    :param prefixes: iterable with prefixes
    :param ipv6: set to True for IPv6 prefixes
    :return: list of filtered out prefixes
    """
    if ipv6:
        pyt = pytricia.PyTricia(128)
    else:
        pyt = pytricia.PyTricia()
    for p in prefixes:
        pyt.insert(p, p)
        if pyt.parent(p):
            del pyt[p]
        elif pyt.children(p):
            for child in pyt.children(p):
                del pyt[child]

    return [pyt[p] for p in pyt]
ippfxs = ["10.0.1.0/24", "10.0.1.128/25", "10.0.1.192/28", "10.0.2.0/24"]
ippfxs_filt = filter_out_subnets(ippfxs)
print(ippfxs_filt)